# ExplainableCropAI System Demonstration

This notebook demonstrates the capabilities of the ExplainableCropAI system for interpretable paddy disease detection. The system combines:

1. **Disease Detection**: Accurate classification of paddy diseases
2. **Visual Explanations**: Highlighting important regions in the image (Grad-CAM)
3. **Uncertainty Quantification**: Estimating model confidence
4. **Natural Language Explanations**: Generating human-readable explanations
5. **Actionable Recommendations**: Context-aware treatment suggestions

Let's begin by setting up the environment and importing the necessary modules.

In [ ]:
# Add the parent directory to the path so we can import from src
import sys
import os
from pathlib import Path

# Add parent directory to path
module_path = str(Path(os.getcwd()).parent)
if module_path not in sys.path:
    sys.path.append(module_path)

# Import required libraries
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import pandas as pd
import seaborn as sns

# Import from our modules
from src.explainable_system import ExplainableCropAI

# Set plotting style
plt.style.use('ggplot')
sns.set_style("whitegrid")

# Display settings
from IPython.display import display, HTML
HTML("<style>.container { width:100% !important; }</style>")

## 1. Initialize the ExplainableCropAI System

First, we'll create an instance of the ExplainableCropAI system. In a production environment, this would load a pre-trained model, but for demonstration purposes, we'll use the model initialized with random weights.

In [ ]:
# Initialize the system
system = ExplainableCropAI(model_path=None)  # In a real scenario, provide path to trained model

print(f"System initialized on device: {system.device}")
print(f"Supported disease classes: {system.class_names}")

## 2. Load Sample Images

For this demonstration, we'll use sample images of paddy diseases. You should download some sample images from the Paddy Doctor dataset and place them in the data directory.

In [ ]:
# Define paths to sample images
sample_images = {
    "Bacterial Leaf Blight": "../data/samples/bacterial_leaf_blight.jpg",
    "Brown Spot": "../data/samples/brown_spot.jpg",
    "Leaf Blast": "../data/samples/leaf_blast.jpg",
    "Healthy": "../data/samples/healthy.jpg"
}

# Check if the files exist
for disease, path in sample_images.items():
    if not os.path.exists(path):
        print(f"Warning: Sample image for {disease} not found at {path}")
        print("Please download sample images or update the paths.")

# Display the sample images if they exist
fig, axes = plt.subplots(1, len(sample_images), figsize=(15, 5))

for i, (disease, path) in enumerate(sample_images.items()):
    if os.path.exists(path):
        img = Image.open(path)
        axes[i].imshow(img)
        axes[i].set_title(disease)
        axes[i].axis('off')
    else:
        axes[i].text(0.5, 0.5, f"Sample not found\n{disease}", ha='center', va='center')
        axes[i].axis('off')

plt.tight_layout()
plt.show()

## 3. Generate Predictions and Explanations

Now, let's use the system to generate predictions and explanations for one of the sample images.

In [ ]:
# Select a sample image
selected_disease = "Bacterial Leaf Blight"
image_path = sample_images.get(selected_disease)

if image_path and os.path.exists(image_path):
    # Generate prediction with explanations
    result = system.predict(image_path)
    
    # Display prediction
    print(f"Predicted disease: {result['prediction']['class_name']}")
    print(f"Confidence: {result['prediction']['confidence']:.4f}")
    
    # Display visual explanation
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].imshow(result['visual_explanation']['original_img'])
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    axes[1].imshow(result['visual_explanation']['overlay'])
    axes[1].set_title("Visual Explanation (Grad-CAM)")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Display natural language explanation
    print("\nExplanation:")
    print(result['nl_explanation']['full_explanation'])
    
    # Display recommendations
    print("\nRecommendations:")
    print(f"Treatment: {result['recommendations']['recommendations']['treatment']}")
    print(f"Prevention: {result['recommendations']['recommendations']['prevention']}")
    
    if result['recommendations']['recommendations']['verification']:
        print(f"Note: {result['recommendations']['recommendations']['verification']}")
else:
    print(f"Sample image for {selected_disease} not found. Please update the path.")

## 4. Visualize the Complete Results

Let's use the built-in visualization function to see a comprehensive view of all the system's outputs.

In [ ]:
if image_path and os.path.exists(image_path):
    # Visualize complete results
    system.visualize_results(result)
else:
    print(f"Sample image for {selected_disease} not found. Please update the path.")

## 5. Exploring Uncertainty

One of the key features of ExplainableCropAI is uncertainty estimation. Let's examine the uncertainty metrics in more detail.

In [ ]:
if 'uncertainty' in result:
    uncertainty = result['uncertainty']
    
    # Create a dataframe for visualization
    metrics = {
        'Metric': ['Entropy', 'Mutual Information', 'Class Agreement'],
        'Value': [uncertainty['entropy'], uncertainty['mutual_information'], uncertainty['class_agreement']]
    }
    
    df = pd.DataFrame(metrics)
    
    # Plot the metrics
    plt.figure(figsize=(10, 5))
    ax = sns.barplot(x='Metric', y='Value', data=df)
    plt.title('Uncertainty Metrics')
    plt.ylabel('Value')
    plt.ylim(0, 1)  # Normalized metrics
    
    # Add value labels
    for i, v in enumerate(df['Value']):
        ax.text(i, v + 0.05, f"{v:.3f}", ha='center')
    
    plt.tight_layout()
    plt.show()
    
    # Visualize class distribution
    if 'class_distribution' in uncertainty:
        class_dist = uncertainty['class_distribution']
        
        # Create a dataframe for the class distribution
        classes = [system.class_names[int(cls)] for cls in class_dist.keys()]
        counts = list(class_dist.values())
        
        dist_df = pd.DataFrame({
            'Class': classes,
            'Count': counts
        })
        
        # Sort by count
        dist_df = dist_df.sort_values('Count', ascending=False)
        
        # Plot
        plt.figure(figsize=(12, 5))
        ax = sns.barplot(x='Class', y='Count', data=dist_df)
        plt.title('Class Distribution Across MC Dropout Samples')
        plt.xticks(rotation=45, ha='right')
        
        # Add value labels
        for i, v in enumerate(dist_df['Count']):
            ax.text(i, v + 0.5, str(v), ha='center')
        
        plt.tight_layout()
        plt.show()

## 6. Comparing Visual Evidence with Explanation

Let's examine how the visual evidence from Grad-CAM aligns with the textual explanation.

In [ ]:
if all(key in result for key in ['visual_explanation', 'nl_explanation', 'visual_features']):
    # Display the visual evidence
    visual_features = result['visual_features']
    visual_evidence = result['nl_explanation']['visual_evidence']
    
    print(f"Visual Features Detected: {', '.join(visual_features)}")
    print(f"Visual Evidence Description: {visual_evidence}")
    
    # Display the Grad-CAM heatmap with the features
    plt.figure(figsize=(10, 6))
    plt.imshow(result['visual_explanation']['overlay'])
    plt.title(f"Visual Evidence: {visual_evidence}")
    plt.axis('off')
    plt.show()

## 7. Exploring Recommendations in Detail

Finally, let's take a deeper look at the recommendations provided by the system.

In [ ]:
if 'recommendations' in result:
    recs = result['recommendations']
    
    # Create a styled HTML display of recommendations
    html = f"""
    <div style="background-color: #f8f9fa; padding: 20px; border-radius: 10px; font-family: Arial, sans-serif;">
        <h2 style="color: #495057;">Disease Management Recommendations</h2>
        <div style="margin-bottom: 15px;">
            <h3 style="margin-bottom: 5px; color: #343a40;">Disease: {recs['disease']}</h3>
            <p><strong>Severity:</strong> <span style="color: {'green' if recs['severity'] == 'low' else 'orange' if recs['severity'] == 'medium' else 'red'}; font-weight: bold;">{recs['severity'].upper()}</span></p>
            <p><strong>Confidence:</strong> {recs['confidence']:.2f}</p>
            <p><strong>Risk Level:</strong> <span style="color: {'green' if recs['risk_level'] == 'low' else 'orange' if recs['risk_level'] == 'medium' else 'red'}; font-weight: bold;">{recs['risk_level'].upper()}</span></p>
        </div>
        
        <div style="background-color: white; padding: 15px; border-radius: 5px; margin-bottom: 15px;">
            <h4 style="color: #343a40; margin-top: 0;">Treatment Recommendations</h4>
            <p>{recs['recommendations']['treatment']}</p>
        </div>
        
        <div style="background-color: white; padding: 15px; border-radius: 5px; margin-bottom: 15px;">
            <h4 style="color: #343a40; margin-top: 0;">Prevention Strategies</h4>
            <p>{recs['recommendations']['prevention']}</p>
        </div>
    """
    
    # Add verification note if present
    if recs['recommendations']['verification']:
        html += f"""
        <div style="background-color: #ffe8d9; padding: 15px; border-radius: 5px; margin-bottom: 15px;">
            <h4 style="color: #d35400; margin-top: 0;">Important Note</h4>
            <p>{recs['recommendations']['verification']}</p>
        </div>
        """
    
    # Add risk factors if present
    if 'supporting_info' in recs and 'risk_factors' in recs['supporting_info'] and recs['supporting_info']['risk_factors']:
        html += f"""
        <div style="background-color: white; padding: 15px; border-radius: 5px;">
            <h4 style="color: #343a40; margin-top: 0;">Risk Factors</h4>
            <ul>
        """
        
        for factor in recs['supporting_info']['risk_factors']:
            html += f"<li>{factor}</li>\n"
        
        html += "</ul>\n</div>"
    
    html += "</div>"
    
    # Display the HTML
    display(HTML(html))

## 8. Conclusion

This notebook has demonstrated the key features of the ExplainableCropAI system:

1. **Multi-level Explanations**: Visual heatmaps combined with natural language descriptions
2. **Uncertainty-Aware Predictions**: Providing confidence metrics with the predictions
3. **Domain-Specific Explanations**: Explanations in the context of agricultural domain knowledge
4. **Actionable Recommendations**: Context-aware treatment and prevention strategies

The system aims to make AI-driven disease detection more transparent, trustworthy, and useful for farmers and agricultural extension officers.